# Measuring H₀ and Dark Energy Parameters from Pantheon+ Supernovae

Independent reproduction of the Hubble tension using 1701 Type Ia supernovae.

Author: Tokosheva Aiym, Bishkek, Kyrgyzstan

## 1. Setup: data and covariance matrix
We load the Pantheon+ catalogue and its full statistical+systematic covariance matrix.

In [ ]:
!pip install emcee corner astropy --quiet
import numpy as np, pandas as pd, matplotlib.pyplot as plt, emcee, corner
from astropy.cosmology import FlatLambdaCDM, FlatwCDM, Flatw0waCDM

u = "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/"
df = pd.read_csv(u+"Pantheon%2BSH0ES.dat", sep=r'\s+')
raw = pd.read_csv(u+"Pantheon%2BSH0ES_STAT%2BSYS.cov", header=None).values.flatten()
N = int(raw[0]); cov = raw[1:].reshape((N,N))
cov = (cov + cov.T)/2                 # remove ~1e-8 rounding asymmetry
cov_inv = np.linalg.inv(cov)
z = df['zHD'].values; mu = df['MU_SH0ES'].values; err = df['MU_SH0ES_ERR_DIAG'].values
print(f"{len(df)} supernovae, covariance {cov.shape}")

## 2. Hubble diagram
Distance modulus versus redshift, compared with late- and early-Universe models.

In [ ]:
m = z > 0.01
zl = np.logspace(np.log10(0.01), np.log10(z.max()), 200)
plt.figure(figsize=(9,6))
plt.scatter(z[m], mu[m], s=5, alpha=0.3, color='gray', label='Pantheon+ SNe Ia')
plt.plot(zl, FlatLambdaCDM(H0=73.5,Om0=0.3).distmod(zl).value,'b-',lw=2,label=r'$H_0=73.5$ (late)')
plt.plot(zl, FlatLambdaCDM(H0=67.4,Om0=0.3).distmod(zl).value,'r--',lw=2,label=r'$H_0=67.4$ (early)')
plt.xscale('log'); plt.xlabel('Redshift $z$'); plt.ylabel(r'Distance modulus $\mu$')
plt.legend(); plt.grid(alpha=0.3); plt.savefig('figures/hubble_diagram.png',dpi=200,bbox_inches='tight'); plt.show()

## 3. Hubble constant with full covariance
χ² = Δᵀ C⁻¹ Δ, scanned over a wide grid.

In [ ]:
def chi2_cov(H0, Om=0.3):
    d = mu - FlatLambdaCDM(H0=H0, Om0=Om).distmod(z).value
    return d @ cov_inv @ d

grid = np.linspace(50, 90, 400)
c2 = np.array([chi2_cov(h) for h in grid])
best = grid[np.argmin(c2)]
within = grid[c2 <= c2.min()+1]; sig = (within.max()-within.min())/2
print(f"H0 = {best:.2f} +- {sig:.2f} km/s/Mpc (statistical only)")

plt.figure(figsize=(8,5))
plt.plot(grid, c2, 'darkred'); plt.axvline(best, color='blue', ls='--', label=f'minimum: {best:.1f}')
plt.xlabel(r'$H_0$ [km s$^{-1}$ Mpc$^{-1}$]'); plt.ylabel(r'$\chi^2$')
plt.legend(); plt.grid(alpha=0.3); plt.savefig('figures/chi2_curve.png',dpi=200,bbox_inches='tight'); plt.show()

## 4. Matter density: H₀–Ωm confidence regions

In [ ]:
H0g = np.linspace(71,76,60); Omg = np.linspace(0.15,0.55,60)
C = np.array([[chi2_cov(h,Om) for h in H0g] for Om in Omg]); C -= C.min()
plt.figure(figsize=(8,6))
cs = plt.contourf(H0g,Omg,C,levels=[0,2.30,6.17,11.8],colors=['#1a5c3a','#2d8659','#7dc4a0'],alpha=0.85)
plt.contour(H0g,Omg,C,levels=[2.30,6.17],colors='white',linewidths=1)
ib,jb = np.unravel_index(np.argmin(C),C.shape); plt.plot(H0g[jb],Omg[ib],'r*',ms=16,label='best fit')
plt.xlabel(r'$H_0$'); plt.ylabel(r'$\Omega_m$'); plt.colorbar(cs,label=r'$\Delta\chi^2$')
plt.legend(); plt.savefig('figures/contour_H0_Om.png',dpi=200,bbox_inches='tight'); plt.show()

## 5. Dark energy equation of state (MCMC)
Sampling H₀, Ωm and w with emcee.

In [ ]:
def logp(p):
    H0,Om,w = p
    if not (60<H0<80 and 0.05<Om<0.6 and -2<w<-0.3): return -np.inf
    d = mu - FlatwCDM(H0=H0,Om0=Om,w0=w).distmod(z).value
    return -0.5*(d @ cov_inv @ d)

sam = emcee.EnsembleSampler(20, 3, logp)
sam.run_mcmc(np.array([73,0.3,-1.0]) + 1e-3*np.random.randn(20,3), 1200, progress=True)
s = sam.get_chain(discard=300, flat=True)
for i,n in enumerate(['H0','Om','w']):
    q = np.percentile(s[:,i],[16,50,84]); print(f"{n} = {q[1]:.3f} (+{q[2]-q[1]:.3f}/-{q[1]-q[0]:.3f})")
fig = corner.corner(s, labels=[r'$H_0$',r'$\Omega_m$',r'$w$'], truths=[None,0.3,-1.0],
                    truth_color='red', quantiles=[0.16,0.5,0.84], show_titles=True, title_fmt='.3f')
fig.savefig('figures/corner_plot.png',dpi=200,bbox_inches='tight')

## 6. Isotropy test
Hemispheric fits along 40 random axes.

In [ ]:
ra = np.radians(df['RA'].values); dec = np.radians(df['DEC'].values)
vec = np.array([np.cos(dec)*np.cos(ra), np.cos(dec)*np.sin(ra), np.sin(dec)]).T

def fit_H0(mask):
    g = np.linspace(65,82,150)
    c = [np.sum(((mu[mask]-FlatLambdaCDM(H0=h,Om0=0.3).distmod(z[mask]).value)/err[mask])**2) for h in g]
    return g[np.argmin(c)]

np.random.seed(0)
d40 = np.random.randn(40,3); d40 /= np.linalg.norm(d40,axis=1,keepdims=True)
diffs = []
for d in d40:
    p = vec @ d
    if (p>0).sum()>50 and (p<=0).sum()>50: diffs.append(fit_H0(p>0)-fit_H0(p<=0))
diffs = np.array(diffs)
print(f"mean |ΔH0| = {np.mean(np.abs(diffs)):.2f}, max = {np.max(np.abs(diffs)):.2f} km/s/Mpc")

## 7. H₀ in redshift shells

In [ ]:
for lo,hi in [(0.01,0.05),(0.05,0.1),(0.1,0.3),(0.3,1.0),(1.0,3.0)]:
    msk = (z>lo)&(z<=hi)
    if msk.sum()>20: print(f"z=[{lo}-{hi}]: H0={fit_H0(msk):.2f} ({msk.sum()} SNe)")